# Artpedia Image Captioning — Full Pipeline (LOCAL)

> **Local version** — runs on your own machine inside the `lavis` conda environment.

Pipeline:
1. **Data prep** — download and cache Artpedia images to local Drive
2. **Fine-tuning** — partially fine-tune BLIP's text decoder on the train split
3. **Evaluation** — BLEU-4 and CIDEr for pretrained vs fine-tuned
4. **t-SNE** — visualise how fine-tuning shifts decoder representations
5. **VizWiz** — zero-shot robustness evaluation on real-world images
6. **MLflow** — notes on viewing logged metrics

## Prerequisites

Assumes the `lavis` conda environment with LAVIS and all dependencies already installed.
Run this notebook **from the notebooks folder** (`mscai-multimodal-project/notebooks`).

```
conda activate lavis
jupyter notebook
```

In [ ]:
# Local paths
DATA_DIR    = "../data"      # where the TRAIN, VAL and TEST splits are cached
OUTPUTS_DIR = "../outputs"   # all run artefacts (checkpoints, eval, t-SNE, MLflow)
CHECKPOINT  = None           # populated by the auto-detect cell after fine-tuning

print(f"DATA_DIR    = {DATA_DIR}")
print(f"OUTPUTS_DIR = {OUTPUTS_DIR}")
print(f"CHECKPOINT  = {CHECKPOINT}")

In [ ]:
import subprocess, sys

def run(script_args):
    """Run a project script with real-time line-by-line output.
    Uses the kernel's own Python so the lavis environment is guaranteed.
    stderr is merged into stdout so warnings/tracebacks appear inline.
    """
    with subprocess.Popen(
        [sys.executable, "-u"] + [str(a) for a in script_args],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f"Script failed (exit code {proc.returncode})")

## 1. Data Preparation

Downloads Artpedia images from Wikimedia (500 px thumbnails) and writes manifests.
Images already on disk are reused — safe to rerun after interruptions.

Run the **test or validation split first** (326 images, good for verifying setup).
The cells are commented out — uncomment to run.
Add `--rebuild-manifest` to regenerate the manifest while reusing cached images.

In [ ]:
# Download and cache the TEST split (339 records).
#run([
#    "../src/prepare_data.py",
#    "--split",      "test",
#    "--output-dir", DATA_DIR,
#    "--delay",      "2.0",
#])

In [ ]:
# Download and cache the VAL split (339 records).
#run([
#    "../src/prepare_data.py",
#    "--split",      "val",
#    "--output-dir", DATA_DIR,
#    "--delay",      "2.0",
#])

In [ ]:
# Download and cache the TRAIN split (2252 records).
# Uncomment and run deliberately once test/val are confirmed working.

# run([
#     "../src/prepare_data.py",
#     "--split",      "train",
#     "--output-dir", DATA_DIR,
#     "--delay",      "2.0",
# ])

# To regenerate the manifest from existing images:
# run(["../src/prepare_data.py", "--split", "train", "--output-dir", DATA_DIR, "--rebuild-manifest"])

## 2. Fine-tuning

Fine-tunes BLIP's **text decoder** (last n transformer layers + cls head) while keeping the **vision encoder frozen**.

Validation loss is computed on `val.jsonl` after every epoch and logged to MLflow alongside training loss.
Checkpoints are saved to `OUTPUTS_DIR/train_<timestamp>/` after each epoch; the best-loss checkpoint is also kept as `blip_artpedia_best.pth`.
Early stopping is active (`--patience p`) — training halts after p consecutive epochs with no improvement in val loss.

In [ ]:
run([
    "../src/train_caption.py",
    "--manifest",          f"{DATA_DIR}/processed/train.jsonl",
    "--val-manifest",      f"{DATA_DIR}/processed/val.jsonl",
    "--base-dir",          DATA_DIR,
    "--output-dir",        OUTPUTS_DIR,
    "--epochs",            "10",
    "--batch-size",        "4",
    "--lr",                "1e-4",
    "--unfreeze-last-n",   "2",
    "--fp16",
    "--num-workers",       "0",
    "--patience",          "3",
    "--use-mlflow",
    "--mlflow-experiment", "blip-artpedia",
])

Auto-detect the latest training run and pick the best checkpoint.
Set EPOCH to a specific integer (e.g. 5) to pick that epoch's checkpoint instead.

In [ ]:
import glob, os

EPOCH = None   # None → blip_artpedia_best.pth;  integer → blip_artpedia_epoch<N>.pth

run_dirs = sorted(glob.glob(OUTPUTS_DIR + "/train_*"))
if not run_dirs:
    print("No training runs found in", OUTPUTS_DIR, "— run the training cell first.")
else:
    latest_run = run_dirs[-1]
    if EPOCH is not None:
        candidate  = os.path.join(latest_run, f"blip_artpedia_epoch{EPOCH}.pth")
        CHECKPOINT = candidate if os.path.exists(candidate) \
                     else os.path.join(latest_run, "blip_artpedia_best.pth")
    else:
        CHECKPOINT = os.path.join(latest_run, "blip_artpedia_best.pth")
    print(f"CHECKPOINT = {CHECKPOINT}")
    print(f"Exists     : {os.path.exists(CHECKPOINT)}")

## 3. Evaluation

Computes **BLEU-4** and **CIDEr** on the test split for both the pretrained base model and the fine-tuned checkpoint.
Results are saved to `eval_results.json` and logged to MLflow.

Update `CHECKPOINT` in the paths cell above to point to the epoch you want to evaluate.

In [ ]:
run([
    "../src/evaluate_caption.py",
    "--manifest",          f"{DATA_DIR}/processed/test.jsonl",
    "--base-dir",          DATA_DIR,
    "--checkpoint",        CHECKPOINT,
    "--split",             "test",
    "--output-dir",        OUTPUTS_DIR,
    "--use-mlflow",
    "--mlflow-experiment", "blip-artpedia",
    "--run-name",          "local-eval",
])

## 4. t-SNE Comparison

Visualises how fine-tuning shifts the **decoder's multimodal representations** (cross-attention hidden states, not raw image embeddings) on the test set, coloured by painting century.

In [ ]:
run([
    "../src/tsne_compare.py",
    "--manifest",   f"{DATA_DIR}/processed/test.jsonl",
    "--base-dir",   DATA_DIR,
    "--checkpoint", CHECKPOINT,
    "--split",      "test",
    "--output-dir", OUTPUTS_DIR,
    "--output",     "tsne_compare.png",
])

In [ ]:
# Display the most recent t-SNE plot inline.
import glob
from IPython.display import Image as IPImage

tsne_dirs = sorted(glob.glob(OUTPUTS_DIR + "/tsne_*"))
if tsne_dirs:
    pngs = sorted(glob.glob(tsne_dirs[-1] + "/*.png"))
    if pngs:
        print(f"Displaying: {pngs[0]}")
        display(IPImage(pngs[0]))
    else:
        print("No PNG found in", tsne_dirs[-1])
else:
    print("No t-SNE runs found in", OUTPUTS_DIR, "— run the t-SNE cell first.")

## 5. Zero-shot Robustness Evaluation on VizWiz

[VizWiz-Caps](https://vizwiz.org/tasks-and-datasets/image-captioning/) contains photos taken by blind people —
a real-world domain-robustness test for the museum-adapted model.

The evaluation set was downloaded locally and stored in `../data/vizwiz/`.
Adjust `--limit` to control how many samples are evaluated.

In [ ]:
run([
    "../src/evaluate_vizwiz.py",
    "--checkpoint",   CHECKPOINT,
    "--limit",        "100",
    "--vizwiz-dir",   "../data/vizwiz",
    "--output-dir",   OUTPUTS_DIR,
    "--use-mlflow",
    "--run-name",     "vizwiz-zeroshot",
])

## 6. MLflow — Viewing Results

MLflow logs to `outputs/mlruns.db` (SQLite). All scripts share the same database so training, evaluation, and VizWiz zero-shot runs appear together.

**Start the UI** — open a terminal in the repo root with `lavis_clean` active:
```
conda activate lavis_clean
python src/mlflow_ui.py
```
Then open [http://127.0.0.1:5000](http://127.0.0.1:5000) in your browser.

**Stop the UI** — in the same terminal press `Ctrl+C`, then run:
```
taskkill /F /IM python.exe /T
```
> This kills all Python processes. If you have other Python processes running (e.g. a training job), stop those first and only use `taskkill` when Python is idle.